# Popularity groups, user profiles, segments, and history buckets

This notebook completes the data-preparation step (Step 1) of the methodology and produces the input tables that the UPD evaluator in `upd-evaluation.ipynb` expects. It uses the **training split only** for every quantity below, so no test information leaks into popularity labels or user profiles.

Set `splitSeed` and `baselineRunId` in the configuration cell before running. It rebuilds the per-user 80/20 split using the same hash-based ordering as `als-baseline.ipynb`, then checks its counts against that ALS run's `split-config.txt`. It does not retrain ALS and does not need the baseline notebook's live session.

**Outputs** (saved under `outputs/popularity-profiles/<run timestamp>/`):

| Table | Columns |
|---|---|
| `movie-popularity-groups` | `movieId`, `ratingCount`, `cumulativeShare`, `popularityGroup` (H/M/T) |
| `user-popularity-profiles` | `userId`, `head`, `mid`, `tail` |
| `user-segments` | `userId`, `segment` |
| `user-history-buckets` | `userId`, `trainingRatingCount`, `historyBucket` |
| `evaluation-users` | `userId` |

In [1]:
val splitSeed = 43L
val baselineRunId = "2026-11-27T18-58-31.586034700Z"
require(Set(42L, 43L, 44L).contains(splitSeed), s"Unsupported split seed: $splitSeed")
require(baselineRunId.nonEmpty, "Set baselineRunId to the matching ALS export.")
println(s"PASS: configured splitSeed=$splitSeed and baselineRunId=$baselineRunId")


PASS: configured splitSeed=43 and baselineRunId=2026-11-27T18-58-31.586034700Z


splitSeed: Long = 43L
baselineRunId: String = "2026-11-27T18-58-31.586034700Z"

## 1. Environment

Only `spark-sql` is needed here; ALS (`spark-mllib`) is not used. The assertions match the runtime verified in the other notebooks.

In [2]:
import $ivy.`org.apache.spark::spark-sql:3.5.9`

import org.apache.spark.sql.SparkSession

Failed to download https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-client-runtime/3.3.4/hadoop-client-runtime-3.3.4-sources.jar
Failed to download https://repo1.maven.org/maven2/io/netty/netty-all/4.1.96.Final/netty-all-4.1.96.Final-sources.jar
Failed to download https://repo1.maven.org/maven2/io/netty/netty-all/4.1.96.Final/netty-all-4.1.96.Final-sources.jar.sha1
Failed to download https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-client-runtime/3.3.4/hadoop-client-runtime-3.3.4-sources.jar.sha1


import $ivy.$
import org.apache.spark.sql.SparkSession

In [3]:
val spark = SparkSession.builder()
  .appName("CS 7265 - Popularity Profiles")
  .master("local[2]")
  .config("spark.driver.host", "127.0.0.1")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
val scalaVersion = scala.util.Properties.versionNumberString
val javaVersion = System.getProperty("java.version")
val checkCount = spark.range(3).count()

assert(spark.version == "3.5.9")
assert(scalaVersion == "2.12.18")
assert(System.getProperty("java.specification.version") == "17")
assert(checkCount == 3L)

println(s"Spark: ${spark.version}")
println(s"Scala: $scalaVersion")
println(s"Java: $javaVersion")
println(s"Environment check passed: $checkCount synthetic rows.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/27 15:31:16 INFO SparkContext: Running Spark version 3.5.9
26/11/27 15:31:16 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/11/27 15:31:16 INFO SparkContext: Java version 17.0.20.1
26/11/27 15:31:18 INFO ResourceUtils: ==============================================================
26/11/27 15:31:18 INFO ResourceUtils: No custom resources configured for spark.driver.
26/11/27 15:31:18 INFO ResourceUtils: ==============================================================
26/11/27 15:31:18 INFO SparkContext: Submitted application: CS 7265 - Popularity Profiles
26/11/27 15:31:19 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: offHeap, amount: 0, script: , vendor: ), task resources: Map(cpus -> name: cpus, amount: 1.0)
26/11/27 15:31:19 INFO Reso

Spark: 3.5.9
Scala: 2.12.18
Java: 17.0.20.1
Environment check passed: 3 synthetic rows.


spark: SparkSession = org.apache.spark.sql.SparkSession@2ef5436b
scalaVersion: String = "2.12.18"
javaVersion: String = "17.0.20.1"
checkCount: Long = 3L

## 2. Load ratings and rebuild the training/test split

The next cells repeat the loading and splitting steps from `als-baseline.ipynb`. The split is deterministic (the shuffle key is `xxhash64(splitSeed, userId, movieId)`), so it reproduces the same training and test sets on any machine.

In [4]:
case class Rating(userId: Int, movieId: Int, rating: Float, timestamp: Long)

class RatingParser extends (String => Rating) with Serializable {
  def apply(line: String): Rating = {
    val fields = line.split("::")
    require(fields.length == 4, "Expected four fields per MovieLens rating")

    Rating(
      fields(0).toInt,
      fields(1).toInt,
      fields(2).toFloat,
      fields(3).toLong
    )
  }
}

defined class Rating
defined class RatingParser

In [5]:
val ratingsPath = Seq(
  "data/ml-1m/ratings.dat",
  "../data/ml-1m/ratings.dat"
).find(path => new java.io.File(path).isFile).getOrElse {
  throw new IllegalArgumentException("Place MovieLens 1M ratings.dat in data/ml-1m/ and run from the repository root or notebooks folder.")
}

val sc = spark.sparkContext
val lines = sc.textFile(ratingsPath)
val ratingsRDD = lines.map(new RatingParser)

import spark.implicits._
val ratingsDF = ratingsRDD.toDF()

ratingsPath: String = "../data/ml-1m/ratings.dat"
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@2f37a4cb
lines: org.apache.spark.rdd.RDD[String] = ../data/ml-1m/ratings.dat MapPartitionsRDD[8] at textFile at cmd5.sc:9
ratingsRDD: org.apache.spark.rdd.RDD[Rating] = MapPartitionsRDD[9] at map at cmd5.sc:10
import spark.implicits._
ratingsDF: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 2 more fields]

In [6]:
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._

val trainingFraction = 0.8

val perUser = Window.partitionBy("userId")
val ratingsWithCounts = ratingsDF.withColumn(
  "userRatingCount",
  count("*").over(perUser)
)

val ratingsWithKeys = ratingsWithCounts.withColumn(
  "shuffleKey",
  xxhash64(lit(splitSeed), col("userId"), col("movieId"))
)

val shuffledPerUser = Window
  .partitionBy("userId")
  .orderBy(col("shuffleKey"), col("movieId"))

val rankedRatings = ratingsWithKeys.withColumn(
  "ratingPosition",
  row_number().over(shuffledPerUser)
)

val ratingsWithCutoff = rankedRatings.withColumn(
  "trainingCount",
  floor(col("userRatingCount") * trainingFraction)
)

val training = ratingsWithCutoff.filter(
  col("ratingPosition") <= col("trainingCount")
)
val test = ratingsWithCutoff.filter(
  col("ratingPosition") > col("trainingCount")
)

val trainingRatings = training.select("userId", "movieId", "rating", "timestamp")
val testRatings = test.select("userId", "movieId", "rating", "timestamp")

import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._
trainingFraction: Double = 0.8
perUser: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@8f2fb54
ratingsWithCounts: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 3 more fields]
ratingsWithKeys: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 4 more fields]
shuffledPerUser: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@c3f8e3c
rankedRatings: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 5 more fields]
ratingsWithCutoff: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 6 more fields]
training: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 6 more fields]
test: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 6 more fields]
trainingRatings: org

### Check that this is the same split as the ALS baseline

The expected train/test counts are read from the selected ALS run's `split-config.txt`; comparing both counts and the seed confirms this rebuild matches the baseline split. The overlap check independently confirms that no user–movie pair appears in both sets.

In [7]:
val trainingRatingCount = trainingRatings.count()
val testRatingCount = testRatings.count()
val trainingUserCount = trainingRatings.select("userId").distinct().count()
val testUserCount = testRatings.select("userId").distinct().count()
val overlapCount = trainingRatings.select("userId", "movieId")
  .intersect(testRatings.select("userId", "movieId")).count()

println(s"Training: $trainingRatingCount ratings from $trainingUserCount users")
println(s"Test: $testRatingCount ratings from $testUserCount users")
println(s"Overlapping user-movie pairs: $overlapCount")

val baselineSplitConfigPath = new java.io.File(
  new java.io.File(ratingsPath).getCanonicalFile.getParentFile.getParentFile.getParentFile,
  s"outputs/als-baseline/$baselineRunId/split-config.txt").toPath
require(java.nio.file.Files.isRegularFile(baselineSplitConfigPath),
  s"Missing split-config.txt for ALS run $baselineRunId; rerun als-baseline.ipynb with the selected seed.")
val baselineSplitConfig = new String(
  java.nio.file.Files.readAllBytes(baselineSplitConfigPath), "UTF-8")
  .split("\\r?\\n").flatMap { line =>
    val separator = line.indexOf('=')
    if (separator > 0) Some(line.substring(0, separator).trim -> line.substring(separator + 1).trim) else None
  }.toMap
val expectedSplitSeed = baselineSplitConfig.getOrElse("splitSeed",
  throw new IllegalArgumentException("ALS split-config.txt has no splitSeed." )).toLong
val expectedTrainingRatings = baselineSplitConfig.getOrElse("trainingRatings",
  throw new IllegalArgumentException("ALS split-config.txt has no trainingRatings." )).toLong
val expectedTestRatings = baselineSplitConfig.getOrElse("testRatings",
  throw new IllegalArgumentException("ALS split-config.txt has no testRatings." )).toLong
assert(expectedSplitSeed == splitSeed,
  s"ALS export uses splitSeed=$expectedSplitSeed, configured seed is $splitSeed.")
assert(trainingRatingCount == expectedTrainingRatings,
  s"Training count $trainingRatingCount differs from ALS export count $expectedTrainingRatings.")
assert(testRatingCount == expectedTestRatings,
  s"Test count $testRatingCount differs from ALS export count $expectedTestRatings.")
assert(trainingUserCount == 6040L && testUserCount == 6040L, "Every user must appear in both sets.")
assert(overlapCount == 0L, "Training and test must not overlap.")
println("Split matches the ALS baseline.")

Training: 797758 ratings from 6040 users
Test: 202451 ratings from 6040 users
Overlapping user-movie pairs: 0
Split matches the ALS baseline.


trainingRatingCount: Long = 797758L
testRatingCount: Long = 202451L
trainingUserCount: Long = 6040L
testUserCount: Long = 6040L
overlapCount: Long = 0L
baselineSplitConfigPath: java.nio.file.Path = C:\Users\navnc\OneDrive\Documents\GitHub\history-aware-popularity-calibration\outputs\als-baseline\2026-11-27T18-58-31.586034700Z\split-config.txt
baselineSplitConfig: Map[String, String] = Map(
  "testRatings" -> "202451",
  "trainingRatings" -> "797758",
  "splitSeed" -> "43",
  "alsSeed" -> "43",
  "trainingFraction" -> "0.8"
)
expectedSplitSeed: Long = 43L
expectedTrainingRatings: Long = 797758L
expectedTestRatings: Long = 202451L

## 3. Label movies Head, Mid, or Tail

Popularity is measured on the training ratings only. Movies are sorted by rating count (most-rated first, ties broken by ascending movie ID so the order is deterministic), and the cumulative share of all training ratings is accumulated down the list.

- **Head (H):** the most-rated movies that together account for the first 20% of ratings. The movie whose rating count carries the cumulative share past 20% is included in Head.
- **Tail (T):** the least-rated movies that together account for the last 20% of ratings, using the same boundary convention from the bottom.
- **Mid (M):** every other movie.

The 20/60/20 split is therefore by share of **ratings**, not share of movies, so Head contains few movies and Tail contains many. Only movies that appear in training receive a label; this covers every movie the ALS model can recommend.

In [8]:
val headShare = 0.20
val tailShare = 0.20

val totalTrainingRatings = trainingRatings.count().toDouble
val movieCounts = trainingRatings.groupBy("movieId").agg(count("*").as("ratingCount"))

val popularityOrder = Window
  .orderBy(col("ratingCount").desc, col("movieId").asc)
  .rowsBetween(Window.unboundedPreceding, Window.currentRow)

val movieGroups = movieCounts
  .withColumn("cumulativeShare",
    sum(col("ratingCount")).over(popularityOrder) / lit(totalTrainingRatings))
  .withColumn("shareBefore",
    col("cumulativeShare") - col("ratingCount") / lit(totalTrainingRatings))
  .withColumn("popularityGroup",
    when(col("shareBefore") < headShare, "H")
      .when(col("cumulativeShare") > 1.0 - tailShare, "T")
      .otherwise("M"))
  .select("movieId", "ratingCount", "cumulativeShare", "popularityGroup")
  .cache()

val groupSummary = movieGroups.groupBy("popularityGroup").agg(
  count("*").as("movieCount"),
  sum("ratingCount").as("ratingSum")
).withColumn("ratingShare", col("ratingSum") / lit(totalTrainingRatings))
 .orderBy("popularityGroup")

groupSummary.show(truncate = false)

+---------------+----------+---------+-------------------+
|popularityGroup|movieCount|ratingSum|ratingShare        |
+---------------+----------+---------+-------------------+
|H              |113       |159597   |0.20005690948884247|
|M              |1068      |478506   |0.5998134772700493 |
|T              |2506      |159655   |0.2001296132411082 |
+---------------+----------+---------+-------------------+



headShare: Double = 0.2
tailShare: Double = 0.2
totalTrainingRatings: Double = 797758.0
movieCounts: org.apache.spark.sql.package.DataFrame = [movieId: int, ratingCount: bigint]
popularityOrder: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@5b1c25ad
movieGroups: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [movieId: int, ratingCount: bigint ... 2 more fields]
groupSummary: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [popularityGroup: string, movieCount: bigint ... 2 more fields]

### Check the labels

Every training movie must have exactly one label, and the three groups must cover every training rating exactly once. Head and Tail each hold at least 20% of ratings, and slightly more because the boundary movie is included in its group. The upper bound of 21% catches mistakes without being sensitive to the exact boundary movie.

In [9]:
val labeledMovieCount = movieGroups.count()
val trainingMovieCount = trainingRatings.select("movieId").distinct().count()
val groupRows = groupSummary.collect().map(r =>
  r.getString(0) -> (r.getLong(1), r.getLong(2), r.getDouble(3))).toMap

assert(labeledMovieCount == trainingMovieCount, "Every training movie needs exactly one label.")
assert(movieGroups.select("movieId").distinct().count() == labeledMovieCount, "Duplicate movie labels.")
assert(groupRows.keySet == Set("H", "M", "T"), "All three groups must be present.")
assert(groupRows.values.map(_._2).sum == trainingRatingCount, "Groups must cover every training rating once.")
assert(groupRows("H")._3 >= headShare - 1e-9 && groupRows("H")._3 < 0.21, "Head share out of range.")
assert(groupRows("T")._3 >= tailShare - 1e-9 && groupRows("T")._3 < 0.21, "Tail share out of range.")

println(s"Labeled $labeledMovieCount movies (H/M/T): " +
  s"${groupRows("H")._1} / ${groupRows("M")._1} / ${groupRows("T")._1}")
println("Movie popularity label checks passed.")

Labeled 3687 movies (H/M/T): 113 / 1068 / 2506
Movie popularity label checks passed.


labeledMovieCount: Long = 3687L
trainingMovieCount: Long = 3687L
groupRows: Map[String, (Long, Long, Double)] = Map(
  "H" -> (113L, 159597L, 0.20005690948884247),
  "M" -> (1068L, 478506L, 0.5998134772700493),
  "T" -> (2506L, 159655L, 0.2001296132411082)
)

## 4. Build user popularity profiles

Each user's profile `p(u)` is the share of their training ratings that falls in Head, Mid, and Tail movies, weighted by the rating value as in the reference paper. A user who rated one Head movie 5 and one Tail movie 3 has weight 5 on Head and 3 on Tail, giving `p(u) = (0.625, 0, 0.375)`. The three shares sum to 1 by construction.

`trainingRatingCount` (the number of training ratings per user, `n_u`) is kept for the history buckets and for the later history-aware extension.

In [10]:
val labeledTraining = trainingRatings
  .join(movieGroups.select("movieId", "popularityGroup"), Seq("movieId"), "inner")

assert(labeledTraining.count() == trainingRatingCount, "Joining labels must not drop or duplicate ratings.")

val ratingValue = col("rating").cast("double")
def groupWeight(label: String) =
  sum(when(col("popularityGroup") === label, ratingValue).otherwise(0.0))

val userStats = labeledTraining.groupBy("userId").agg(
  sum(ratingValue).as("ratingTotal"),
  groupWeight("H").as("headWeight"),
  groupWeight("M").as("midWeight"),
  groupWeight("T").as("tailWeight"),
  count("*").as("trainingRatingCount")
)

val userProfiles = userStats.select(
  col("userId"),
  (col("headWeight") / col("ratingTotal")).as("head"),
  (col("midWeight") / col("ratingTotal")).as("mid"),
  (col("tailWeight") / col("ratingTotal")).as("tail"),
  col("trainingRatingCount")
).cache()

val badProfiles = userProfiles.filter(
  abs(col("head") + col("mid") + col("tail") - 1.0) > 1e-9 ||
  col("head") < 0.0 || col("mid") < 0.0 || col("tail") < 0.0
).count()

assert(userProfiles.count() == 6040L, "Every user needs a profile.")
assert(badProfiles == 0L, "Profiles must be nonnegative and sum to 1.")

userProfiles.agg(
  avg("head").as("meanHead"), avg("mid").as("meanMid"), avg("tail").as("meanTail"),
  min("trainingRatingCount").as("minTrainingRatings"),
  max("trainingRatingCount").as("maxTrainingRatings")
).show(truncate = false)
println("User profile checks passed.")

+------------------+------------------+------------------+------------------+------------------+
|meanHead          |meanMid           |meanTail          |minTrainingRatings|maxTrainingRatings|
+------------------+------------------+------------------+------------------+------------------+
|0.2910710035324173|0.5641647039780939|0.1447642924894888|16                |1851              |
+------------------+------------------+------------------+------------------+------------------+

User profile checks passed.


labeledTraining: org.apache.spark.sql.package.DataFrame = [movieId: int, userId: int ... 3 more fields]
ratingValue: org.apache.spark.sql.Column = CAST(rating AS DOUBLE)
defined function groupWeight
userStats: org.apache.spark.sql.package.DataFrame = [userId: int, ratingTotal: double ... 4 more fields]
userProfiles: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, head: double ... 3 more fields]
badProfiles: Long = 0L

## 5. Assign user segments and history buckets

**Segments.** Users are ranked by Head share (highest first, ties broken by user ID) and cut into three equal-sized groups with `ntile(3)`: the top third is *blockbuster-focused*, the middle third *diverse*, and the bottom third *niche-focused*. With 6,040 users the sizes are 2,014, 2,013, and 2,013.

**History buckets.** Based on each user's number of training ratings: *short* (fewer than 50), *medium* (50 to 200 inclusive), and *long* (more than 200).

In [11]:
val segmentOrder = Window.orderBy(col("head").desc, col("userId").asc)

val userSegments = userProfiles
  .withColumn("tile", ntile(3).over(segmentOrder))
  .withColumn("segment",
    when(col("tile") === 1, "blockbuster-focused")
      .when(col("tile") === 2, "diverse")
      .otherwise("niche-focused"))
  .select("userId", "segment", "head")
  .cache()

val historyBuckets = userProfiles.select(
  col("userId"),
  col("trainingRatingCount"),
  when(col("trainingRatingCount") < 50, "short")
    .when(col("trainingRatingCount") <= 200, "medium")
    .otherwise("long").as("historyBucket")
).cache()

println("Segments (by Head share):")
userSegments.groupBy("segment").agg(
  count("*").as("userCount"),
  min("head").as("minHead"), avg("head").as("meanHead"), max("head").as("maxHead")
).orderBy(col("meanHead").desc).show(truncate = false)

println("History buckets:")
historyBuckets.groupBy("historyBucket").agg(
  count("*").as("userCount"),
  min("trainingRatingCount").as("minN"), max("trainingRatingCount").as("maxN")
).orderBy("minN").show(truncate = false)

println("Segment x history bucket (user counts):")
userSegments.join(historyBuckets, Seq("userId"))
  .groupBy("segment").pivot("historyBucket", Seq("short", "medium", "long"))
  .count().orderBy("segment").show(truncate = false)

val segmentSizes = userSegments.groupBy("segment").count().collect().map(_.getLong(1))
assert(userSegments.count() == 6040L && historyBuckets.count() == 6040L, "Every user needs a segment and bucket.")
assert(segmentSizes.length == 3 && segmentSizes.max - segmentSizes.min <= 1L, "Segments must be equal in size.")
println("Segment and bucket checks passed.")

Segments (by Head share):
+-------------------+---------+-------------------+-------------------+-------------------+
|segment            |userCount|minHead            |meanHead           |maxHead            |
+-------------------+---------+-------------------+-------------------+-------------------+
|blockbuster-focused|2014     |0.3311897106109325 |0.4438652033766844 |0.8769230769230769 |
|diverse            |2013     |0.21782178217821782|0.27048156468041834|0.33079847908745247|
|niche-focused      |2013     |0.0                |0.15879033881444504|0.21782178217821782|
+-------------------+---------+-------------------+-------------------+-------------------+

History buckets:
+-------------+---------+----+----+
|historyBucket|userCount|minN|maxN|
+-------------+---------+----+----+
|short        |2205     |16  |49  |
|medium       |2617     |50  |200 |
|long         |1218     |201 |1851|
+-------------+---------+----+----+

Segment x history bucket (user counts):
+------------------

segmentOrder: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@61167138
userSegments: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, segment: string ... 1 more field]
historyBuckets: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, trainingRatingCount: bigint ... 1 more field]
segmentSizes: Array[Long] = Array(2014L, 2013L, 2013L)

## 6. Export the evaluator inputs

The tables are written under `outputs/popularity-profiles/<run timestamp>/` (ignored by Git), together with a short `split-config.txt` recording the split settings, so the files can be traced to the split that produced them. The read-back check confirms the row count of every table. The evaluation users are taken from the training users, so that table must have 6,040 rows.

In [12]:
val repositoryRoot = new java.io.File(ratingsPath).getCanonicalFile
  .getParentFile.getParentFile.getParentFile
val profileRunId = java.time.Instant.now().toString.replace(":", "-")
val profileDirectory = new java.io.File(
  repositoryRoot, s"outputs/popularity-profiles/$profileRunId")

def writeTable(df: org.apache.spark.sql.DataFrame, name: String): String = {
  val path = new java.io.File(profileDirectory, name).getPath
  df.coalesce(1).write.mode("errorifexists").parquet(path)
  path
}

val evaluationUsers = trainingRatings.select("userId").distinct()

val exportPaths = Map(
  "movie-popularity-groups" -> writeTable(movieGroups, "movie-popularity-groups"),
  "user-popularity-profiles" -> writeTable(userProfiles.select("userId", "head", "mid", "tail"), "user-popularity-profiles"),
  "user-segments" -> writeTable(userSegments.select("userId", "segment"), "user-segments"),
  "user-history-buckets" -> writeTable(historyBuckets, "user-history-buckets"),
  "evaluation-users" -> writeTable(evaluationUsers, "evaluation-users")
)

val configText = Seq(
  s"baselineRunId=$baselineRunId",
  s"splitSeed=$splitSeed",
  s"trainingFraction=$trainingFraction",
  s"trainingRatings=$trainingRatingCount",
  s"testRatings=$testRatingCount",
  s"headShare=$headShare",
  s"tailShare=$tailShare",
  "historyBuckets=short<50;medium=50-200;long>200",
  "segments=ntile(3) by head share descending, ties by userId"
).mkString("\n") + "\n"
java.nio.file.Files.write(
  new java.io.File(profileDirectory, "split-config.txt").toPath,
  configText.getBytes("UTF-8"))

val expectedRows = Map(
  "movie-popularity-groups" -> trainingMovieCount,
  "user-popularity-profiles" -> 6040L,
  "user-segments" -> 6040L,
  "user-history-buckets" -> 6040L,
  "evaluation-users" -> 6040L
)
expectedRows.foreach { case (name, expected) =>
  val actual = spark.read.parquet(exportPaths(name)).count()
  assert(actual == expected, s"$name has $actual rows; expected $expected.")
  println(s"Verified $name: $actual rows")
}

println(s"Export folder: ${profileDirectory.getPath}")

Verified evaluation-users: 6040 rows
Verified user-popularity-profiles: 6040 rows
Verified user-history-buckets: 6040 rows
Verified user-segments: 6040 rows
Verified movie-popularity-groups: 3687 rows
Export folder: C:\Users\navnc\OneDrive\Documents\GitHub\history-aware-popularity-calibration\outputs\popularity-profiles\2026-11-27T20-34-44.309675700Z


repositoryRoot: java.io.File = C:\Users\navnc\OneDrive\Documents\GitHub\history-aware-popularity-calibration
profileRunId: String = "2026-11-27T20-34-44.309675700Z"
profileDirectory: java.io.File = C:\Users\navnc\OneDrive\Documents\GitHub\history-aware-popularity-calibration\outputs\popularity-profiles\2026-11-27T20-34-44.309675700Z
defined function writeTable
evaluationUsers: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int]
exportPaths: Map[String, String] = Map(
  "evaluation-users" -> "C:\\Users\\navnc\\OneDrive\\Documents\\GitHub\\history-aware-popularity-calibration\\outputs\\popularity-profiles\\2026-11-27T20-34-44.309675700Z\\evaluation-users",
  "user-popularity-profiles" -> "C:\\Users\\navnc\\OneDrive\\Documents\\GitHub\\history-aware-popularity-calibration\\outputs\\popularity-profiles\\2026-11-27T20-34-44.309675700Z\\user-popularity-profiles",
  "user-history-buckets" -> "C:\\Users\\navnc\\OneDrive\\Documents\\GitHub\\history-aware-popularity-calibratio

## 7. Stop the Spark session

Run this cell when finished. The exported tables are what `upd-evaluation.ipynb` loads in its "Use real inputs" section.

In [13]:
movieGroups.unpersist(blocking = true)
userProfiles.unpersist(blocking = true)
userSegments.unpersist(blocking = true)
historyBuckets.unpersist(blocking = true)

spark.stop()
println("Spark session stopped.")

Spark session stopped.


res13_0: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [movieId: int, ratingCount: bigint ... 2 more fields]
res13_1: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, head: double ... 3 more fields]
res13_2: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, segment: string ... 1 more field]
res13_3: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, trainingRatingCount: bigint ... 1 more field]